# End-to-End Video Summarizer

**Pipeline:** Video → Extract audio → Speech-to-text (Whisper) → Clean transcript → Summarize (BART)

This notebook builds the pipeline step by step, so you can see and test each stage on its own before chaining them together at the end.

Each section below has:
- a short explanation of *what* the stage does and *why*
- a code cell implementing it
- a quick test so you can see the output before moving on


## Step 0 — Setup

Two things to install before running this notebook:

1. **ffmpeg** (system tool, not a Python package)
   - Windows: download from ffmpeg.org and add to PATH
   - Mac: `brew install ffmpeg`
   - Linux: `sudo apt install ffmpeg`
2. **Python packages** — run the cell below once.

The first time you run Whisper/BART below, they'll each download their model weights (a few GB total). This only happens once.

In [ ]:
# Run once to install dependencies (safe to re-run, pip will skip what's already installed)
%pip install openai-whisper transformers torch


In [ ]:
import os
import subprocess
import tempfile

import whisper
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

print("Imports OK")


## Step 1 — Extract audio from the video

Video files bundle audio + video streams together, but speech recognition only needs the sound. We use `ffmpeg` to pull out a mono, 16kHz `.wav` audio track — that's the format Whisper expects internally, so we convert here rather than making Whisper do it.

In [ ]:
def extract_audio(video_path: str, output_audio_path: str) -> str:
    """Extract mono 16kHz audio track from a video file using ffmpeg."""
    command = [
        "ffmpeg", "-y",
        "-i", video_path,
        "-ac", "1",
        "-ar", "16000",
        "-vn",
        output_audio_path,
    ]
    result = subprocess.run(command, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"ffmpeg failed:\n{result.stderr}")
    return output_audio_path


In [ ]:
# --- Test it ---
VIDEO_PATH = "your_video.mp4"  # <-- change this to your video file
AUDIO_PATH = "extracted_audio.wav"

extract_audio(VIDEO_PATH, AUDIO_PATH)
print(f"Audio extracted to {AUDIO_PATH}")


## Step 2 — Speech-to-text with Whisper

Whisper is an encoder-decoder Transformer: the *encoder* compresses the audio's spectrogram into a rich numerical representation, and the *decoder* generates text one word-piece at a time from that representation — conceptually similar to a translation model, just translating "audio" into "text." It was trained on 680,000 hours of multilingual audio, so it works well with no fine-tuning needed.

`model_size` controls the accuracy/speed trade-off: `tiny` < `base` < `small` < `medium` < `large`. Start with `base`.

In [ ]:
def transcribe_audio(audio_path: str, model_size: str = "base") -> str:
    """Transcribe audio to text using OpenAI Whisper."""
    print(f"Loading Whisper model ({model_size})...")
    model = whisper.load_model(model_size)
    print("Transcribing... this can take a while on CPU.")
    result = model.transcribe(audio_path)
    return result["text"].strip()


In [ ]:
# --- Test it ---
raw_transcript = transcribe_audio(AUDIO_PATH, model_size="base")
print(raw_transcript[:1000])  # preview the first part


## Step 3 — Clean the transcript

Raw ASR output is usually a single wall of text. This step just tidies up whitespace so it's ready for the summarizer.

In [ ]:
def clean_transcript(text: str) -> str:
    """Light cleanup: collapse extra whitespace from the raw ASR output."""
    return " ".join(text.split())


In [ ]:
# --- Test it ---
transcript = clean_transcript(raw_transcript)
print(f"Transcript length: {len(transcript)} characters")


## Step 4 — Summarize with BART

BART is also an encoder-decoder Transformer, but pretrained differently: during pretraining, text is deliberately corrupted (words deleted, shuffled, masked) and the model learns to reconstruct the original — this builds a deep understanding of language structure. It's then fine-tuned on article → summary pairs, so it *generates new sentences* rather than just copying from the transcript (this is called **abstractive** summarization).

BART has a ~1024 token input limit, so long transcripts (e.g. a 30-minute lecture) are split into chunks, each summarized separately, then combined and summarized once more.

In [ ]:
def summarize_text(text: str, max_length: int = 150, min_length: int = 40) -> str:
    """Summarize text using a pretrained BART model.

    Note: transformers v5 removed the old pipeline("summarization", ...)
    shortcut, so we load the tokenizer and model directly and call
    generate() ourselves instead.
    """
    model_name = "facebook/bart-large-cnn"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

    def _summarize_chunk(chunk: str) -> str:
        inputs = tokenizer(chunk, return_tensors="pt", truncation=True, max_length=1024)
        summary_ids = model.generate(
            inputs["input_ids"],
            max_length=max_length,
            min_length=min_length,
            num_beams=4,
            early_stopping=True,
        )
        return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

    max_chunk_chars = 3000
    chunks = [text[i:i + max_chunk_chars] for i in range(0, len(text), max_chunk_chars)]

    summaries = []
    for i, chunk in enumerate(chunks):
        print(f"Summarizing chunk {i + 1}/{len(chunks)}...")
        summaries.append(_summarize_chunk(chunk))

    combined_summary = " ".join(summaries)

    if len(chunks) > 1:
        print("Combining chunk summaries into one final summary...")
        return _summarize_chunk(combined_summary)

    return combined_summary


In [ ]:
# --- Test it ---
summary = summarize_text(transcript)
print(summary)


## Step 5 — Put it all together

Now that each stage works on its own, chain them into a single function that goes straight from a video file to a summary.

In [ ]:
def run_pipeline(video_path: str, whisper_model: str = "base") -> dict:
    """Run the full video -> transcript -> summary pipeline end to end."""
    with tempfile.TemporaryDirectory() as tmp_dir:
        audio_path = os.path.join(tmp_dir, "audio.wav")

        print("Step 1/4: Extracting audio from video...")
        extract_audio(video_path, audio_path)

        print("Step 2/4: Transcribing speech to text...")
        raw = transcribe_audio(audio_path, whisper_model)

        print("Step 3/4: Cleaning transcript...")
        clean = clean_transcript(raw)

        print("Step 4/4: Generating summary...")
        summary = summarize_text(clean)

    return {"transcript": clean, "summary": summary}


In [ ]:
# --- Run the whole thing end to end ---
results = run_pipeline(VIDEO_PATH, whisper_model="base")

print("TRANSCRIPT:\n", results["transcript"][:500], "...\n")
print("SUMMARY:\n", results["summary"])


## Next steps

- **Evaluation:** measure Word Error Rate (WER) for the transcript against a known-correct reference, and ROUGE score for the summary against a reference summary — this is how you'd back up "it works" with numbers for your report.
- **Speaker diarization:** if you need "who said what," that's a separate model (e.g. `pyannote-audio`) layered on top of Whisper.
- **Deployment:** wrap `run_pipeline()` in a simple Streamlit or Flask app for a demo-ready UI.
